In [48]:
# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# **Part 1: Fuel Consumption Based on Horse Power**

In [49]:
# Loading and inspecting the dataset
hp_df = pd.read_csv('/content/FuelEconomy.csv')

print("Shape:", hp_df.shape)
print("\nColumns:")
print(hp_df.columns.tolist())

display(hp_df.head())

display(hp_df.describe(include="all"))
display(hp_df.isna().sum())

Shape: (100, 2)

Columns:
['Horse Power', 'Fuel Economy (MPG)']


,Horse Power,Fuel Economy (MPG)
0,118.770799,29.344195
1,176.326567,24.695934
2,219.262465,23.952010
3,187.310009,23.384546
4,218.594340,23.426739


,Horse Power,Fuel Economy (MPG)
count,100.000000,100.000000
mean,213.676190,23.178501
std,62.061726,4.701666
min,50.000000,10.000000
25%,174.996514,20.439516
50%,218.928402,23.143192
75%,251.706476,26.089933
max,350.000000,35.000000


,0
Horse Power,0
Fuel Economy (MPG),0


In [50]:
# Helper functions
# Handling missing data
def missing_values(input_df, target_col):
  """Drops all missing rows before splitting the dataset into X and Y."""
  clean_df = input_df.dropna().copy()
  X = clean_df.drop(columns=[target_col])
  y = clean_df[target_col]
  return X, y

# Splitting data
def split_data(X, y, test_size=0.3, random_state=42):
  """Splits the dataset into 70% training, 30% testing."""
  return train_test_split(X, y, test_size=test_size, random_state=random_state)

# For computing metrics
def compute_metrics(y_true, y_pred):
  """Returns the MSE, MAE, and R^2."""
  return {
      "MSE": mean_squared_error(y_true, y_pred),
      "MAE": mean_absolute_error(y_true, y_pred),
      "R^2": r2_score(y_true, y_pred)
  }

# Running models
def run_models_and_evaluate(df_in, degrees=(1, 2, 3, 4), target_col=target_col,
                            test_size=0.30, random_state=42):
    """Train/evaluate linear (deg=1) + polynomial regression models.
        Returns a DataFrame of metrics.
    """
    X, y = missing_values(df_in, target_col=target_col)
    X_train, X_test, y_train, y_test = split_data(X, y, test_size=test_size, random_state=random_state)

    rows = []

    for deg in degrees:
        if deg == 1:
            model = LinearRegression()
            model_name = "Linear Regression"
        else:
            model = Pipeline([
                ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
                ("lr", LinearRegression())
            ])
            model_name = f"Polynomial Regression (degree={deg})"

        # Fit model
        model.fit(X_train, y_train)

        # Predict
        yhat_train = model.predict(X_train)
        yhat_test  = model.predict(X_test)

        # Metrics
        train_m = compute_metrics(y_train, yhat_train)
        test_m  = compute_metrics(y_test, yhat_test)

        rows.append({
            "Model": model_name,
            "Train MSE": train_m["MSE"],
            "Train MAE": train_m["MAE"],
            "Train R^2": train_m["R^2"],
            "Test MSE": test_m["MSE"],
            "Test MAE": test_m["MAE"],
            "Test R^2": test_m["R^2"]
        })

    return pd.DataFrame(rows)

In [51]:
# Running model on horsepower dataset
hp_results = run_models_and_evaluate(
    df_in=hp_df,
    degrees=(1, 2, 3, 4),
    target_col='Horse Power',
)

display(hp_results)

,Model,Train MSE,Train MAE,Train R^2,Test MSE,Test MAE,Test R^2
0,Linear Regression,357.699180,16.061689,0.906320,318.561087,14.940628,0.912561
1,Polynomial Regression (degree=2),350.879731,15.995824,0.908106,331.105434,15.148330,0.909118
2,Polynomial Regression (degree=3),345.108668,15.746762,0.909618,318.404012,14.764973,0.912604
3,Polynomial Regression (degree=4),339.700171,15.508465,0.911034,313.798757,14.735471,0.913868


## **Analysis**
1. The Polynomial Regression (degree=4) model performed the best. From the table above, it has the highest Test R^2 value (closest to 1, explains the most variance in horse power), as well as the lowest Test MSE and Test MAE (smallest average prediction error in horse power).

2. Although the degree=4 model performed the best, increasing polynomial degree doesn't always improve performance. From the linear model to the degree=2 model, performance decreased; this is evident from the degree=2 model's lower Test R^2 value and higher Test MSE and Test MAE values compared to the linear model. Moving from degree=2 to degree=3 then improved the performance again (higher Test R^2, lower Test MSE and Test MAE). Degree=4 performs better than degree=3, but only by a small margin (slight increase in Test R^2 and slight decrease in Test MSE and Test MAE).

3. None of the models performed unexpectedly poorly.

# **Part 2: Electricity Consumption Based on Weather**

In [52]:
# Loading and inspecting the dataset
elec_df = pd.read_csv('/content/electricity_consumption_based_weather_dataset.csv')

print("Shape:", elec_df.shape)
print("\nColumns:")
print(elec_df.columns.tolist())

display(elec_df.head())

display(elec_df.describe(include="all"))
display(elec_df.isna().sum())

Shape: (1433, 6)

Columns:
['date', 'AWND', 'PRCP', 'TMAX', 'TMIN', 'daily_consumption']


,date,AWND,PRCP,TMAX,TMIN,daily_consumption
0,2006-12-16,2.5,0.0,10.6,5.0,1209.176
1,2006-12-17,2.6,0.0,13.3,5.6,3390.460
2,2006-12-18,2.4,0.0,15.0,6.7,2203.826
3,2006-12-19,2.4,0.0,7.2,2.2,1666.194
4,2006-12-20,2.4,0.0,7.2,1.1,2225.748


,date,AWND,PRCP,TMAX,TMIN,daily_consumption
count,1433,1418.000000,1433.000000,1433.000000,1433.000000,1433.000000
unique,1433,NaN,NaN,NaN,NaN,NaN
top,2010-11-26,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN
mean,NaN,2.642313,3.800488,17.187509,9.141242,1561.078061
std,NaN,1.140021,10.973436,10.136415,9.028417,606.819667
min,NaN,0.000000,0.000000,-8.900000,-14.400000,14.218000
25%,NaN,1.800000,0.000000,8.900000,2.200000,1165.700000
50%,NaN,2.400000,0.000000,17.800000,9.400000,1542.650000
75%,NaN,3.300000,1.300000,26.100000,17.200000,1893.608000


,0
date,0
AWND,15
PRCP,0
TMAX,0
TMIN,0
daily_consumption,0


In [53]:
# 'Date' is a string column; the models can only train on numeric features.
elec_df = elec_df.drop(columns=["date"])

elec_results = run_models_and_evaluate(
    df_in=elec_df,
    degrees=(1, 2, 3, 4),
    target_col='daily_consumption'
)

display(elec_results)

,Model,Train MSE,Train MAE,Train R^2,Test MSE,Test MAE,Test R^2
0,Linear Regression,272403.396174,384.465016,0.276000,2.481258e+05,375.404537,0.299333
1,Polynomial Regression (degree=2),264765.769932,379.648753,0.296300,2.552685e+05,379.039083,0.279163
2,Polynomial Regression (degree=3),259249.534870,375.952901,0.310961,2.656237e+05,385.235167,0.249922
3,Polynomial Regression (degree=4),251909.339001,372.116566,0.330470,1.215149e+07,578.642201,-33.313844


## **Analysis**
1. The linear regression model generalizes best, achieving the highest Test R^2 value (~0.3) along with the lowest Test MAE and MSE out of all the models. This result suggests that while weather variables explain some variability in electricity consumption, the overall relationship is relatively weak based on the dataset.

2. Polynomial regression models do not consistently improve test performance when compared to linear regression. Although the Train R^2 values show a consistent increase from the linear regression model up to the degree=4 model, the Test R^2 value shows a steady decrease.

3. The higher-degree polynomials in this test set tend towards overfitting. As the polynomial degree increases, training error decreases and Train R^2 increases, while test error increases sharply. Despite the highest Train R^2 value in the degree=4 model, its Test R^2 value is strongly negative, and its Test MSE increases by orders of magnitude. This suggests that the model might be fitting noise in the training data rather than learning generalizable patterns; also known as overfitting.